# v2.3 Test Weighting


## Environment Setup

In [ ]:
import numpy as np
import sys
sys.path.append('..')
import pickle
import logging
import os.path as osp
import tensorflow as tf
from moisture_rnn_pkl import pkl2train
from moisture_rnn import RNNParams, RNNData, RNN, rnn_data_wrap
from utils import hash2, read_yml, read_pkl, retrieve_url, Dict, print_dict_summary, print_first, str2time, logging_setup
from moisture_rnn import RNN
import reproducibility
from data_funcs import rmse, to_json, combine_nested, subset_by_features, read_and_clean
from moisture_models import run_augmented_kf
from metrics import ros_3wind
import copy
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import time

In [ ]:
logging_setup()

In [ ]:
filename = "fmda_rocky_202403-05_f05.pkl"
retrieve_url(
    url = f"https://demo.openwfm.org/web/data/fmda/dicts/{filename}", 
    dest_path = f"../data/{filename}")

In [ ]:
file_paths = [f'../data/{filename}']

In [ ]:
# # read/write control
# train_file='../data/train.pkl'
# train_create=True   # if false, read
# train_write=False
# train_read=False

In [ ]:
# Params used for data filtering
params_data = read_yml("../params_data.yaml") 
params_data

In [ ]:
# Params used for setting up RNN
params = read_yml("../params.yaml", subkey='rnn') 

In [ ]:
feats = ['Ed', 'Ew', 'solar', 'wind', 'elev', 'lon', 'lat', 'rain']
params.update({'features_list': feats})

In [ ]:
train = read_and_clean(file_paths, atm_source="HRRR", params_data = params_data, verbose=True)
train = combine_nested(train)

In [ ]:
# if train_create:
#     params_data.update({'hours': 1440})
#     logging.info('creating the training cases from files %s',file_paths)
#     # osp.join works on windows too, joins paths using \ or /
#     train = process_train_dict(file_paths, atm_dict = "RAWS", params_data = params_data, verbose=True)
#     train = subset_by_features(train, feats)
#     train = combine_nested(train)
# if train_write:
#     with open(train_file, 'wb') as file:
#         logging.info('Writing the rain cases into file %s',train_file)
#         pickle.dump(train, file)
# if train_read:
#     logging.info('Reading the train cases from file %s',train_file)
#     train = read_pkl(train_file)

## Train Models

In [ ]:
reproducibility.set_seed(123)

In [ ]:
params = RNNParams(params)

params.update({
    'hidden_layers': ['dense', 'lstm', 'conv1d', 'dense'],
    'hidden_units': [64, 32, 32, 16],
    'hidden_activation': ['relu','tanh', 'relu', 'relu'],
    'batch_schedule_type': None,
    'stateful': False,
    'early_stopping_patience':15
})

In [ ]:
import importlib
import moisture_rnn
importlib.reload(moisture_rnn)
from moisture_rnn import RNNData

In [ ]:
rnn_dat_sp = rnn_data_wrap(train, params)

In [ ]:
reproducibility.set_seed(123)
rnn0 = RNN(params)
m0, errs0 = rnn0.run_model(rnn_dat_sp)
print(f"{errs0.mean()=}")

### Custom Loss

In [ ]:
params.update({
    'w_alpha': 0.0367 
})

In [ ]:
reproducibility.set_seed(123)
rnn1 = RNN(params)
m1, errs1 = rnn1.run_model(rnn_dat_sp)
print(f"{errs1.mean()=}")

## Compare ROS

In [ ]:
ros_truth = ros_3wind(rnn_dat_sp.y_test)
ros0 = ros_3wind(m0)
ros1 = ros_3wind(m1)

In [ ]:
from utils import rmse_3d

In [ ]:
ros_truth.shape

In [ ]:
ros1.shape

In [ ]:
rmse_3d(ros_truth, ros0).mean()

In [ ]:
rmse_3d(ros_truth, ros1).mean()

In [ ]:
np.all(ros_truth[0,:,0] == ros_3wind(rnn_dat_sp.y_test[0,:,0]))

In [ ]:
# Pairwise Percent Diff (relative to standard MSE baseline)
((rmse_3d(ros_truth, ros0)- rmse_3d(ros_truth, ros1))/rmse_3d(ros_truth, ros0)).mean()

In [ ]:
## Confirm element-wise division below
# ((rmse_3d(ros_truth, ros0)- rmse_3d(ros_truth, ros1))/rmse_3d(ros_truth, ros0))
# (rmse_3d(ros_truth, ros0)- rmse_3d(ros_truth, ros1))[0] / rmse_3d(ros_truth, ros0)[0]